In [14]:
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

import pvlib

import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

import Model_func as mf
import boto3

from dotenv import load_dotenv
import os

load_dotenv()

True

# Collect Data

In [3]:
data_prod_path = '../../data/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
prod_data = mf.data_collection_prod(data_prod_path)

data_sat_path = '../../data/LandSat/result_EarthExplorer_region_ARA.csv'
sat_data = mf.data_coll_landsat(data_sat_path)

data_weather_path = '../../data/openweathermap/merge_openweathermap_cleaned.csv'
weather_data = mf.data_coll_weather(data_weather_path)


In [4]:
targeted_sat_data = mf.add_target_column_sat(sat_data, prod_data)

In [5]:
dfs_by_city = mf.split_data_weather_by_city(weather_data)
merged_weather_data = mf.merge_weather_dfs_by_city(dfs_by_city)

merged_weather_data.shape


(1770, 91)

In [6]:
targeted_weather_data = mf.add_target(merged_weather_data, prod_data)
targeted_weather_data.shape

(1453, 92)

In [16]:
targeted_weather_data.columns

Index(['Moulins_dt', 'Moulins_sunrise', 'Moulins_sunset', 'Moulins_temp',
       'Moulins_feels_like', 'Moulins_pressure', 'Moulins_humidity',
       'Moulins_dew_point', 'Moulins_clouds', 'Moulins_wind_speed',
       'Moulins_wind_deg', 'Moulins_rain', 'Moulins_snow', 'Moulins_city',
       'Moulins_lat', 'Moulins_lon', 'Moulins_weather_main',
       'Moulins_weather_desc', 'Time', 'Aurillac_dt', 'Aurillac_sunrise',
       'Aurillac_sunset', 'Aurillac_temp', 'Aurillac_feels_like',
       'Aurillac_pressure', 'Aurillac_humidity', 'Aurillac_dew_point',
       'Aurillac_clouds', 'Aurillac_wind_speed', 'Aurillac_wind_deg',
       'Aurillac_rain', 'Aurillac_snow', 'Aurillac_city', 'Aurillac_lat',
       'Aurillac_lon', 'Aurillac_weather_main', 'Aurillac_weather_desc',
       'Saint-Étienne_dt', 'Saint-Étienne_sunrise', 'Saint-Étienne_sunset',
       'Saint-Étienne_temp', 'Saint-Étienne_feels_like',
       'Saint-Étienne_pressure', 'Saint-Étienne_humidity',
       'Saint-Étienne_dew_point',

In [10]:
#---MLFlow params
os.environ["APP_URI"] = "https://renergies99-mlflow.hf.space/"
EXPERIMENT_NAME = "first_weather_models"

mlflow.set_tracking_uri(os.environ["APP_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

mlflow.sklearn.autolog()  # enables automatic logging for scikit-learn

#---Data for ML
features = ['temp', 'pressure', 'humidity', 'clouds', 'tch_solaire_(%)']
cols_to_keep = [col for col in targeted_weather_data.columns if col.endswith(tuple(features))]
if not cols_to_keep:
    raise ValueError(f"No column found ending with {features}")
df_temp = targeted_weather_data[cols_to_keep]

df_clean = df_temp.dropna()

target = 'tch_solaire_(%)'
y = df_clean[target].to_numpy() #target

X = df_clean.drop(target, axis=1)

numeric_cols = X.select_dtypes(include='number').columns.tolist()
object_cols = X.select_dtypes(exclude='number').columns.tolist()

transformers = [('num', StandardScaler(), numeric_cols)]
if object_cols:  
    transformers.append(('obj', 'passthrough', object_cols))

preprocessor = ColumnTransformer(transformers=transformers)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('estimator', LinearRegression())
])

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

run_description = f"Features used: {features}\nTarget: {target}"

with mlflow.start_run(experiment_id=experiment.experiment_id, description=run_description):
    # Fit the pipeline (preprocessing + model)
    pipeline.fit(x_train, y_train)
    
    # Score
    score = pipeline.score(x_test, y_test)
    mlflow.log_metric("ScoreR2", score)
    
    # Log the full pipeline as a model
    mlflow.sklearn.log_model(pipeline, artifact_path="pipeline_model")
    
print(f"R2 score: {score}")



2025/11/18 11:01:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\hardy\anaconda3\envs\Jedi\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/11/18 11:01:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\hardy\anaconda3\envs\Jedi\Lib\site-p

🏃 View run legendary-lark-70 at: https://renergies99-mlflow.hf.space/#/experiments/4/runs/d2ecdc920874415aaf123d7d81e1cd5f
🧪 View experiment at: https://renergies99-mlflow.hf.space/#/experiments/4
R2 score: 0.6229306430543733


In [11]:
preprocessor = pipeline.named_steps['preprocessor']

feature_names = []

for name, transformer, cols in preprocessor.transformers:
    if name == 'num':
        feature_names.extend(cols)  # StandardScaler ne change pas le nombre de colonnes
    elif name == 'obj':
        feature_names.extend(cols)  # passthrough garde les colonnes telles quelles

# Récupérer les coefficients du modèle
coefs = pipeline.named_steps['estimator'].coef_

print(len(feature_names))
print(len(coefs))


20
20


In [12]:
df_coef = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs
})
df_coef = df_coef.sort_values(by='coefficient', key=abs)

px.bar(df_coef, x='coefficient', y='feature')

In [ ]:
data_weather_path = '../../data/openweathermap/merge_openweathermap_cleaned.csv'
data_prod_path = '../../data/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
features = ['temp', 'pressure', 'humidity', 'clouds']
# possibles features = []
target = ['tch_solaire_(%)']


#----DATA COLLECTION--------------
collected_weather_data = mf.data_collection_weather(data_weather_path)
collected_prod_data = mf.data_collection_prod(data_prod_path)


targeted_weather_data = mf.add_target(collected_weather_data, collected_prod_data)


In [20]:
weather_data = mf.data_coll_weather(data_weather_path)

In [18]:
targeted_weather_data.head()

,Moulins_dt,Moulins_sunrise,Moulins_sunset,Moulins_temp,Moulins_feels_like,Moulins_pressure,Moulins_humidity,Moulins_dew_point,Moulins_clouds,Moulins_wind_speed,...,Nyons_wind_speed,Nyons_wind_deg,Nyons_rain,Nyons_snow,Nyons_city,Nyons_lat,Nyons_lon,Nyons_weather_main,Nyons_weather_desc,tch_solaire_(%)
0,2021-01-09 11:00:00,2021-01-09 07:29:19,2021-01-09 16:17:54,1.11,-2.22,1020,83,-1.28,100,3.06,...,1.90,10,NaN,NaN,Nyons,44.361193,5.140874,Clouds,overcast clouds,13.28
1,2021-01-10 11:00:00,2021-01-10 07:28:56,2021-01-10 16:19:05,0.71,-2.78,1022,73,-3.16,37,3.16,...,1.59,7,NaN,NaN,Nyons,44.361193,5.140874,Clouds,overcast clouds,15.04
2,2021-01-11 11:00:00,2021-01-11 07:28:31,2021-01-11 16:20:18,-0.74,-2.53,1027,79,-3.56,70,1.46,...,2.74,11,NaN,NaN,Nyons,44.361193,5.140874,Clouds,few clouds,16.81
3,2021-01-12 11:00:00,2021-01-12 07:28:03,2021-01-12 16:21:32,4.55,0.86,1023,92,3.36,100,4.83,...,3.99,347,NaN,NaN,Nyons,44.361193,5.140874,Clouds,broken clouds,9.24
4,2021-01-13 11:00:00,2021-01-13 07:27:32,2021-01-13 16:22:48,8.50,6.07,1025,97,8.05,100,4.19,...,5.85,339,NaN,NaN,Nyons,44.361193,5.140874,Clouds,few clouds,7.82


 - plot de chaque variable pour voir si lineaire ou si fonction "usuelle"
 - travailler sur les erreurs
 - ajouter la colonne sunset-sunrise

